###  **1.4. Grouping and aggregating data in Spark**

This demonstration will show how to perform grouping and aggregatino operations using NYC Taxi tip data. We'll sxplore basic grouping, multiple aggregatinos, and windows functions.

####  **Objectives**
* Understand basic grouping operations in Spark
* Perform time-based analysis using aggregations
* Implement complex aggregations with multiple metrics
* Use windows functions for advanced analytics

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Create a the first SparkSession
spark = SparkSession.builder.appName("Basic ETL").getOrCreate()

In [ ]:
# Read the flights data from a CSV file and print the first 5 rows
# trips_df = spark.read.csv("../resources/nyc_taxi_trip_duration.csv", header=True, inferSchema=True)

trips_df = spark.read.parquet("../resources/yellow_tripdata_2025-11.parquet")

trips_df.show(10)

In [ ]:

trips_renamed_df = trips_df.withColumn("pickup_zip", col("PULocationID")).drop("PULocationID")

trips_df.show(10)

####  **B. Basic Grouping Operations**

Let's start with simple grouping operations to understand trip patterns buy location.

In [ ]:
# Count trip by pickup location, to show top 5 most popular pickup locations.

location_count = trips_renamed_df \
    .groupBy("pickup_zip") \
    .count() \
    .orderBy(desc("count"))

print(location_count.show(10))

####  **C. Combining Multiple Aggregations**

Let's perform multiple aggregations by location using the `agg()` method

In [ ]:
# Perform multiple aggregations by location, order by most popular pickup locations.

location_stats = trips_renamed_df \
    .groupBy("pickup_zip") \
    .agg(
        count("*").alias("total_trips"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(sum("fare_amount"), 2).alias("total_fare_amt")
    ) \
    .orderBy(desc("total_trips"))

location_stats.show(5)

####  **D. Windows Function**

Now let's use windows functions for more advanced analytics.

In [ ]:
from pyspark.sql.window import Window

# Create window specs for different ranking methods 
window_by_trips = Window.orderBy(desc("total_trips"))
window_by_fare = Window.orderBy(desc("avg_fare"))

# Add different types of rankings
ranked_locations = location_stats \
    .withColumn("trips_rank", rank().over(window_by_trips)) \
    .withColumn("fare_rank", rank().over(window_by_fare)) \
    .withColumn("fare_quintile", ntile(5).over(window_by_fare)) # Divide into 5 groups by fare

In [ ]:
# Displaying the results.
ranked_locations.select(
    "pickup_zip",
    "total_trips",
    "avg_fare",
    "avg_distance",
    "trips_rank",
    "fare_rank",
    "fare_quintile"
).limit(15).show(15)